# Step 1: Install required packages

In [1]:
!pip install google-auth-oauthlib google-api-python-client

     |████████████████████████████████| 14.9 MB 2.0 MB/s eta 0:00:01
     |████████████████████████████████| 240 kB 14.6 MB/s eta 0:00:01
     |████████████████████████████████| 91 kB 33.6 MB/s eta 0:00:01
     |████████████████████████████████| 173 kB 8.5 MB/s eta 0:00:01
     |████████████████████████████████| 297 kB 55.7 MB/s eta 0:00:01
     |████████████████████████████████| 50 kB 25.7 MB/s eta 0:00:01
     |████████████████████████████████| 181 kB 6.5 MB/s eta 0:00:01
     |████████████████████████████████| 7.2 MB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 180 kB 10.4 MB/s eta 0:00:01
     |████████████████████████████████| 118 kB 25.7 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 56.0 MB/s eta 0:00:01
     |████████████████████████████████| 83 kB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 160 kB 12.0 MB/s eta 0:00:01
You should consider upgrading via the '/Users/talant/Desktop/RAG-Gmail/.venv/bin/python3 -m pip install --upgr

# Step 2: Authenticate with Gmail API

Before running this, you need:
1. Go to https://console.cloud.google.com
2. Create a new project
3. Enable **Gmail API**
4. Go to **Credentials** → Create **OAuth 2.0 Client ID** (Desktop app)
5. Download the JSON and save it as `credentials.json` in this folder

In [23]:
import os
import json
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_gmail_service():
    creds = None
    # token.json stores the access/refresh tokens after first login
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    # If no valid credentials, let the user log in
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)

        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

service = get_gmail_service()
print("Connected to Gmail!")

Connected to Gmail!


# Step 3: Fetch emails

In [27]:
import base64
import re

def get_email_body(payload):
    """Extract body from email payload, preferring plain text but falling back to HTML."""
    plain = None
    html = None

    def extract_parts(part):
        nonlocal plain, html
        mime = part.get('mimeType', '')
        if mime == 'text/plain':
            data = part['body'].get('data', '')
            if data:
                plain = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        elif mime == 'text/html':
            data = part['body'].get('data', '')
            if data:
                html = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        elif mime.startswith('multipart/'):
            for subpart in part.get('parts', []):
                extract_parts(subpart)

    extract_parts(payload)

    if plain:
        return plain
    if html:
        # Strip HTML tags to get readable text
        text = re.sub(r'<[^>]+>', ' ', html)
        text = re.sub(r'&[a-zA-Z]+;', ' ', text)  # decode HTML entities
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    return ''

def fetch_emails(service, max_results=100):
    """Fetch the latest emails from inbox."""
    emails = []
    results = service.users().messages().list(
        userId='me', labelIds=['INBOX'], maxResults=max_results
    ).execute()
    
    messages = results.get('messages', [])
    print(f"Fetching {len(messages)} emails...")
    
    for msg in messages:
        msg_data = service.users().messages().get(
            userId='me', id=msg['id'], format='full'
        ).execute()
        
        headers = {h['name']: h['value'] for h in msg_data['payload']['headers']}
        body = get_email_body(msg_data['payload'])
        
        emails.append({
            'id': msg['id'],
            'subject': headers.get('Subject', 'No Subject'),
            'from': headers.get('From', 'Unknown'),
            'date': headers.get('Date', ''),
            'body': body[:2000]
        })
    
    return emails

emails = fetch_emails(service, max_results=100)
print(f"\nFetched {len(emails)} emails successfully!")
print(f"\nSample email:")
print(f"From: {emails[0]['from']}")
print(f"Subject: {emails[0]['subject']}")
print(f"Date: {emails[0]['date']}")
print(f"Body preview: {emails[0]['body'][:200]}")

Fetching 84 emails...

Fetched 84 emails successfully!

Sample email:
From: Tetiana Grebeniuk via Docusign <dse@docusign.net>
Subject: Completed: Action needed by KHALDAROV, TALANTBEK: Please sign your tax documents
Date: Sat, 28 Mar 2026 13:57:03 -0700
Body preview: Hello Talantbek Khaldarov,    

All parties have completed Action needed by KHALDAROV, TALANTBEK: Please sign your tax documents.
    
    Dear KHALDAROV, TALANTBEK,
Please review and sign your t


# Step 4: Save emails to JSON for later use

In [28]:
with open('emails.json', 'w') as f:
    json.dump(emails, f, indent=2)

print(f"Saved {len(emails)} emails to emails.json")

Saved 84 emails to emails.json
